In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -U peft

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
from tqdm import tqdm
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import random
from transformers import Trainer, AutoTokenizer, TrainingArguments, DataCollatorForSeq2Seq, BitsAndBytesConfig, AutoModelForCausalLM 
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score

torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
modelclass_answers = ["GPT-5",
    "GPT-4",
    "GPT-4 Omni",     
    "DeepSeek-V3",
    "PaLM-2",
    "GPT-3.5",
    "GPT-4 Turbo",
    "Falcon-180B",
    "Claude 3 Opus",
    "Gemini-Pro",
    "GLM",
    "Goliath",
    "Command R",
    "Llama3"]

def make_prompt_llmclass(text, tokenizer):
  prompt = """Task: Identify the language model the given text is generated with. Choose your answer from the given answer choices.
  Answer choices: 
    GPT-5
    GPT-4
    GPT-4 Omni     
    DeepSeek-V3
    PaLM-2
    GPT-3.5
    GPT-4 Turbo
    Falcon-180B
    Claude 3 Opus
    Gemini-Pro
    GLM
    Goliath
    Command R
    Llama3
  Text: """ + text + """
  Answer: """

  system_prompt = 'You are an expert in text classification'
    
  user_prompt = f"""Task: Identify the language model the given text is generated with. Write your answer in one word, choose it from the given answer choices.
Answer choices:
    GPT-5
    GPT-4
    GPT-4 Omni     
    DeepSeek-V3
    PaLM-2
    GPT-3.5
    GPT-4 Turbo
    Falcon-180B
    Claude 3 Opus
    Gemini-Pro
    GLM
    Goliath
    Command R
    Llama3
Text: {text}
Answer:"""


  columns = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]

  full_text = tokenizer.apply_chat_template(columns, tokenize=False)
  return full_text

In [ ]:
def make_prompt(text, tokenizer):
  prompt = """Task: Identify whether the given text is generated by an Language model or is human-written. Choose your answer from the given answer choices.
  Answer choices: 
      LM
      Human
      Unsure
  Text: """ + text + """
  Answer: """

  system_prompt = 'You are an expert in text classification'
    
  user_prompt = f"""Identify whether the given text is generated by an Language model or is human-written. Write your answer in one word, choose it from the given answer choices.
Answer choices:
    LM
    Human
    Unsure
Text: {text}
Answer:"""

  columns = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]

  full_text = tokenizer.apply_chat_template(columns, tokenize=False)
  return full_text

In [ ]:
def add_shots(text, df_aux, tokenizer, num_shots=1):
    df_human = df_aux.loc[(df_aux['generated'] == 0)].sample(frac=1, random_state=42)
    df_lm = df_aux.loc[(df_aux['generated'] == 1)].sample(frac=1, random_state=42)
    system_prompt = 'You are an expert in text classification'
    user_prompt = f"""Identify whether the given text is generated by a Language model or is human-written. Write your answer in one word, choose it from the given answer choices.
Answer choices:
    LM
    Human
    Unsure
Examples:"""

    for i in range(num_shots):
        user_prompt += f"""Text: {df_human['text'][df_human.index[i]]}
Answer: Human
Text: {df_lm['text'][df_lm.index[i]]}
Answer: LM
"""

    user_prompt += f"""Text for classification: {text}
Single word answer:"""

    columns = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]
    full_text = tokenizer.apply_chat_template(columns, tokenize=False)

    return full_text

In [ ]:
def get_model_score(model, tokenizer, dataset, df_aux, num_shots=0, repeat_treshold=8):
  seed_everything(42)
  num_correct = 0
  num_samles = 0
  answers = []
  outofmem = []

  counter = 0
  model.eval()
  for i in tqdm(dataset.index):        
      counter += 1
      if num_shots == 0:
          prompt = make_prompt(dataset['text'][i], tokenizer)#make_prompt_llmclass(dataset['text'][i], tokenizer)
      else:
          prompt = add_shots(dataset['text'][i], df_aux, tokenizer, num_shots=num_shots)

      text_prompt_len = len(prompt)
      inputs = tokenizer(prompt, return_tensors="pt")
      input_ids = inputs.input_ids.to("cuda:0")
      prompt_len = input_ids.shape[1]
      form_answ_position = -1
      repeat_counter = 0

      try:
          while form_answ_position == -1:
              with torch.no_grad():
                generated_ids = model.generate(input_ids, max_length=prompt_len+15)
                generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                prediction = generated_text[generated_text.rfind('answer:'):len(generated_text)]
        
                form_answ_position = max(prediction.find('Human'), prediction.find('human'),
                                        prediction.find('LM'), prediction.find('lm'))
                  
                #for answer in modelclass_answers:
                #    form_answ_position = prediction.find(answer)
                #    if form_answ_position != -1:
                #        break
                
                if form_answ_position != -1:  
                    answers.append(prediction) 
                if repeat_counter > repeat_treshold:
                    answers.append(prediction)
                    break
                else:
                  repeat_counter += 1
      except:
          answers.append('Unsure')
          outofmem.append(i)

  return answers, outofmem

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
      "Qwen/Qwen3-8B",
      device_map="cuda:0",
      quantization_config=quantization_config,
      trust_remote_code=True
  )
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B", trust_remote_code=True)

In [ ]:
model = PeftModel.from_pretrained(model,
    "/kaggle/input/models/ivanchukaev/quen3bft1875/transformers/default/1/checkpoint-1875",
    device_map="cuda:0",
    trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/models/ivanchukaev/quen3bft1875/transformers/default/1/checkpoint-1875",
    trust_remote_code=True)

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/ivanchukaev/testdata/test5000.csv")
df_val = pd.read_csv("/kaggle/input/datasets/ivanchukaev/gcd-validate5k/validate5000.csv")

In [ ]:
modelnames_conv = {'Claude 3 Opus' : "Claude 3 Opus",
                   'Command R+' : "Command R",
                   'DeepSeek-V3.2' : "DeepSeek-V3",
                   'Falcon-180B' : "Falcon-180B",
                   'GLM-130B' : "GLM",
                   'GPT-3.5' : "GPT-3.5",
                   'GPT-4' : "GPT-4",
                   'GPT-5' : "GPT-5",
                   'GPT4 Omni' : "GPT-4 Omni",
                   'GPT4 Turbo 2024-04-09' : "GPT-4 Turbo",
                   'Gemini-Pro' : "Gemini-Pro",
                   'Goliath-120B' : "Goliath",
                   'Llama3 70B' : "Llama3",
                   'PaLM-2' : "PaLM-2",
                   'human' : "human",
                  'YandexGPT 3 Pro' : 'YandexGPT 3 Pro'}
#df['model'] = [modelnames_conv[model] if model in modelnames_conv.keys() else model for model in df['model']]

In [ ]:
answers, outofmem = get_model_score(model, tokenizer, df, df_val, num_shots=2)

answers_filtered = ['Human' if answer.find('Human') != -1 else 'LM' if answer.find('LM') != -1 else answer.replace('Answer:\nassistant\n<think>\n\n</think>\n\n', '') for answer in answers]
print(np.unique(answers_filtered))
answers_count = {answer : 0 for answer in np.unique(answers_filtered)}
for answer in answers_filtered:
  answers_count[answer] += 1
print(answers_count)
eval_data = {'Human-correct' : 0,
       'LM-correct' : 0,
       'Human-wrong' : 0,
       'LM-wrong' : 0,
       'Human-unsure' : 0,
       'LM-unsure' : 0}

counter = 0
correct = 0

for i in df_mod.index:
    if df_mod['generated'][i] == 0:
        Label = 'Human'
    else:
        Label = 'LM'
    Cat = ""
    if not answers_filtered[counter] in ['Human', 'LM']:
        Cat = 'unsure'
    elif Label == answers_filtered[counter]:
        Cat = 'correct'
        correct += 1
    elif Label != answers_filtered[counter]:
        Cat = 'wrong'
    if Cat != "":
        eval_data[Label + "-" + Cat] += 1
    counter += 1

print(correct / counter)
eval_data